# Encrypted GPT-2 Generation

We test the real inference as a service scenario of a chatbot: a prompt is encoded, encrypted and sent to the server that process it autoregressively generating and answer. All encrypted even the argmax feedback.

The caches filling pipeline runs classical baby step giant step packing, while decoding runs Cachemir [1] for maximum slots utilization.


[1] Yu et al., "Cachemir: Fully Homomorphic Encrypted Inference of Generative Large Language Model with KV Cache," arXiv:2602.11470, 2026.

<!-- prereq-cell -->
> **Prerequisites.** This notebook loads artifacts it does not create: the exported
> weights (`weights.bin.zip`, `client.npz`), the calibrated `configs.json`, and, for the
> planned path, a bootstrap plan. Run **`setup_artifacts.ipynb`** first if you do not
> have them.

In [ ]:
import json, os, time
from pathlib import Path

import numpy as np
import torch                                   # torch before _core (NCCL load order)
from transformers import GPT2LMHeadModel, GPT2TokenizerFast

from perseus import _core

# ── write your prompt here ────────────────────────────────────────────────────
PROMPT = ("The history of computing begins long before the first electronic machine, "
          "with mechanical devices built to automate arithmetic, the patient work of "
          "human calculators, and the slow realisation that calculation itself could "
          "be described precisely enough for a machine to carry it out.")
# ──────────────────────────────────────────────────────────────────────────────
P = int(os.environ.get("GEN_PROMPT", "8"))     # prompt tokens (P+N must stay <= 32 for this notebook's purpose)
N = int(os.environ.get("GEN_TOKENS", "10"))    # tokens to generate

tok = GPT2TokenizerFast.from_pretrained("gpt2")
hf = GPT2LMHeadModel.from_pretrained("gpt2").eval()

ids = tok(PROMPT)["input_ids"]
if len(ids) < P:
    raise ValueError(f"PROMPT is {len(ids)} tokens, need >= GEN_PROMPT={P}")
ids = ids[:P]
print(f"prompt ({P} tokens): {tok.decode(ids)!r}")

/leonardo_work/IscrC_eff-SAM2/azirilli/he-aware-training/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]


Loading weights:   4%|▍         | 6/148 [00:00<00:03, 36.50it/s]


Loading weights:  11%|█         | 16/148 [00:00<00:02, 58.38it/s]


Loading weights:  22%|██▏       | 32/148 [00:00<00:01, 91.98it/s]

...


Loading weights:  64%|██████▍   | 95/148 [00:01<00:00, 73.08it/s]


Loading weights:  82%|████████▏ | 121/148 [00:01<00:00, 106.07it/s]


Loading weights:  96%|█████████▌| 142/148 [00:01<00:00, 129.02it/s]


Loading weights: 100%|██████████| 148/148 [00:01<00:00, 99.02it/s] 

prompt (8 tokens): 'The history of computing begins long before the'


## Client side: embed the prompt

`wte[token] + wpe[position]` of the initial prompt is performed client side.

In [2]:
with torch.no_grad():
    emb = (hf.transformer.wte(torch.tensor(ids)) +
           hf.transformer.wpe(torch.arange(P))).numpy()

rows = emb.tolist() + [[0.0] * emb.shape[1] for _ in range(N)]   # tail unused (argmax feedback)
inp_path = Path(os.environ.get("TMPDIR", "/tmp")) / "gpt2_custom_prompt.json"
inp_path.write_text(json.dumps({"inp": rows}))
os.environ["CLASSIFY_INPUT_JSON"] = str(inp_path)
print(f"embedded {P} tokens x {emb.shape[1]} -> {inp_path}")

# plaintext reference: greedy continuation from the same prompt
with torch.no_grad():
    ref = hf.generate(torch.tensor([ids]), max_new_tokens=N, do_sample=False,
                      pad_token_id=tok.eos_token_id)[0, P:].tolist()
print(f"plaintext continuation: {tok.decode(ref)!r}")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


embedded 8 tokens x 768 -> /leonardo_scratch/large/userexternal/azirilli/tmp.50530468/gpt2_custom_prompt.json


plaintext continuation: ' invention of the computer. The first computer was a'


## Encrypted cache prefill → generate

`RunConfig.from_env()` picks up the plan directory.

In [ ]:
cfg = _core.RunConfig.from_env()
cfg.gen_prompt, cfg.gen_tokens, cfg.teacher_forced = P, N, False # False for autoregressive generation
cfg.tokens = P + N

print(f"plan: {os.environ.get('FHE_BOOTSTRAP_PLACEMENTS_DIR', '(eager)').split('/')[-1]}  "
      f"(P+N = {P + N} positions, must be <= 32)")

inputs = _core.read_teacher_forced_inputs(cfg)
t0 = time.perf_counter()
res = _core.run_generate(cfg, inputs) # exploits cuda residency
assert not res.threw, res.error
dt = time.perf_counter() - t0

gen_ids = [int(t) for t in res.top1]
print(f"\ngenerated {res.completed} tokens in {dt:.0f}s ({dt/max(res.completed,1):.0f}s/tok)")
print(f"unplanned_bts = {res.unplanned_bts}")

plan: planned_gpt2_gen  (P+N = 18 positions, must be <= 32)


[ckks_env] recognized environment knobs (effective values):
  LOGN                  = 16  [env]
  CKKS_DEPTH            = 11  [def]
  BTP_DEPTH_OVERHEAD    = 16  [def]
  FIRST_MOD_BITS        = 60  [def]
  BTP_SCALE_BITS        = 53  [def]
  SCALE_BITS            = 58  [def]
  CORRECTION_FACTOR     = 0  [def]
  NUM_LARGE_DIGITS      = 7  [def]
  AUTO_BTS_LEVEL        = 24  [env]
  H_WEIGHT              = 192  [def]
  BTS_ITERATIONS        = 1  [env]
  BTS_PRECISION         = 12  [def]
  CKKS_COMPLEX          = 1  [env]
  SPARSE_BTS_SLOTS      = 0  [def]
  LEVEL_BUDGET          = 4:3  [def]
  limb_chain            = (uniform 53)  [def]


[ckks_params] ===== CKKS crypto context =====
  logN=16  ringDim=65536  slots=32768
  depth=11  btp_overhead=16  mult_depth=27  limbs=28
  first_mod=60  scale(active)=53  (btp_scale=53 scale_bits=58)  logQ~=1491
  bootstrap=on  correction_factor=0  bts_iters=1  bts_prec=12  auto_bts_level=24
  secret=SPARSE_ENCAPSULATED h=192 (OpenFHE accounts UNIFORM_TERNARY)  dnum=7  complex_payload=1
  security: HEStd_128_classic enforced & OK (logQ 1491 <= uniform-ternary budget 1761)
[ckks_params] ===============================


GPU 0: NVIDIA A100-SXM-64GB
 SMs: 124, SharedMem: 164 KB, Blocks/SM: 32, Threads/SM: 2048, L2 size: 32 MB, Bus: 4096-bit


[rot_band] band=11 full_keep=76 (precom entries=1)


...

[gentime] pos=15 e2e=54.7s decode=45.4s argmax=9.3s


[tokstat] tok16 free=4.42GB bts=5523(+512) relevels=1050(+0) enc_cache=309(+0) miss=493(+1) lnf=18/1


[cutmax_bts] entry=1 i0=10 i1=16 i2=24 i3=24 i4=17 sum=3 total=95



generated 10 tokens in 1018s (102s/tok)
unplanned_bts = 0   (necessary, not sufficient — see above)


[generate] pos=16 cutmax=257 fhe_argmax=257 OK gt_argmax=640 z_mass=0.9839 z_off_sum=3.241e-03 z_off_max=3.152e-03@3562 argmax=9.3s
[gentime] pos=16 e2e=10.1s decode=0.8s argmax=9.3s


## The continuation

Decryption happens only now, back on the client. The encrypted run feeds back its own argmax each step, so any
divergence from the plaintext model compounds matching for the first tokens and then
drifting is the expected behaviour, not a failure. The noise introduced by CKKS approximation and encoding itself might lead to different outputs.

In [4]:
print(f"prompt           : {tok.decode(ids)!r}")
print(f"FHE continuation : {tok.decode(gen_ids)!r}")
print(f"ref continuation : {tok.decode(ref)!r}")
agree = sum(a == b for a, b in zip(gen_ids, ref))
print(f"\ntoken agreement  : {agree}/{len(ref)}")
print(f"full FHE text    : {tok.decode(ids + gen_ids)!r}")
assert res.completed == N
assert res.unplanned_bts == 0, f"unplanned bootstraps fired: {res.unplanned_bts}"

prompt           : 'The history of computing begins long before the'
FHE continuation : ' invention of the computer. The first computer was a'
ref continuation : ' invention of the computer. The first computer was a'

token agreement  : 10/10
full FHE text    : 'The history of computing begins long before the invention of the computer. The first computer was a'


## What this showed

- **Encrypted GPT-2** generated a continuation from a prompt of your choosing, with the
  sampled token never leaving the ciphertext domain (encrypted argmax feedback).
- It ran **planned** on `planned_gpt2_gen` across prefill and decode 

Change `PROMPT` in the first cell to prompt it differently, it must be at least
`GEN_PROMPT` tokens.

`GEN_PROMPT + GEN_TOKENS` must stay within **32 total positions** for this notebook purpose. The real limit is the 1024 tokens of GPT2 context (assuming a recapture for prefilling, generation runs endless from a token0 capture).